# Week 03 — Model Creation (Classification)
**Course:** Machine Learning & Deep Learning Project  
**Dataset:** MovieLens (Movie Recommendation System)  

---

### SOP Task List (Verbatim):
> **Week 3: Model Creation**
> - [x] Find out appropriate algorithm for training for your dataset.
> - [x] Use Library for project also.
> - [x] Implement selected Algorithm without use of Library.
> - *User Constraint: Use Classification framing.*


## Week Objective
We will frame the Recommendation task as a **Binary Classification Problem**: *Will the user like this movie?* 
We define "Liked" ($1$) as a rating $\ge 4.0$, and "Disliked" ($0$) as $< 4.0$. 
We will engineer basic features (User Historical Average Rating, Movie Global Average Rating) to train our models.
We will implement **Logistic Regression** twice:
1. Using the `scikit-learn` library.
2. Written entirely from **scratch** using pure `NumPy` mathematics.


In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from IPython.display import Image, display

# Robust path resolution
base_parent = ".." if os.path.exists(os.path.join("..", "CSVs")) else "."
def resolve_file(rel_path):
    for candidate in [os.path.join("..", rel_path), rel_path]:
        if os.path.exists(candidate):
            return candidate
    return os.path.join(base_parent, rel_path)

TRAIN_PATH = resolve_file(os.path.join("data", "processed", "train.csv"))
MODELS_DIR = os.path.join(base_parent, "models")
SRC_DIR = os.path.join(base_parent, "src", "models")
FIGURES_DIR = os.path.join(base_parent, "reports", "figures")

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Random Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment configured successfully.")
print(f"Train Path: {TRAIN_PATH}")
print(f"Models Dir: {MODELS_DIR}")



Environment configured successfully.


---
## 1. Framing Recommendation as Classification
We threshold the continuous ratings to create binary labels.


In [2]:
# Load training data
train_df = pd.read_csv(TRAIN_PATH)

# Create binary classification target
# 1 = Liked (rating >= 4.0), 0 = Not Liked (rating < 4.0)
train_df['is_liked'] = (train_df['rating'] >= 4.0).astype(int)

print(f"Dataset Size: {len(train_df):,}")
print("\nTarget Distribution (is_liked):")
print(train_df['is_liked'].value_counts(normalize=True) * 100)


Dataset Size: 826,848

Target Distribution (is_liked):
is_liked
1    54.421974
0    45.578026
Name: proportion, dtype: float64


---
## 2. Feature Engineering for Logistic Regression
Logistic Regression cannot natively understand categorical IDs (`userId`, `movieId`) without massive one-hot encoding (which crashes RAM). Instead, we replace them with dense features:
- `user_avg_rating`: The historical average rating given by the user.
- `movie_avg_rating`: The historical average rating received by the movie.


In [3]:
# Calculate Averages (Leakage prevented because this is calculated ONLY on train set)
user_avgs = train_df.groupby('userId')['rating'].mean().rename('user_avg_rating')
movie_avgs = train_df.groupby('movieId')['rating'].mean().rename('movie_avg_rating')

# Merge features into training set
train_df = train_df.join(user_avgs, on='userId')
train_df = train_df.join(movie_avgs, on='movieId')

# Prepare X and y matrix
X_train = train_df[['user_avg_rating', 'movie_avg_rating']].values
y_train = train_df['is_liked'].values

print("Feature Matrix Shape:", X_train.shape)
print("First 3 rows of X_train (user_avg, movie_avg):\n", X_train[:3])


Feature Matrix Shape: (826848, 2)
First 3 rows of X_train (user_avg, movie_avg):
 [[2.64935065 4.08392351]
 [3.60616438 3.77826633]
 [3.80701754 3.85785061]]


---
## 3. Algorithm Implementation 1: Using Library (Scikit-Learn)
We train the baseline model using `sklearn.linear_model.LogisticRegression`.


In [4]:
# Initialize and fit library model
model_lib = LogisticRegression(random_state=RANDOM_STATE)
model_lib.fit(X_train, y_train)

# Predict on training set
y_pred_lib = model_lib.predict(X_train)
acc_lib = accuracy_score(y_train, y_pred_lib)

print(f"Library Logistic Regression Training Accuracy: {acc_lib * 100:.2f}%")
print(f"Learned Weights: {model_lib.coef_[0]}")
print(f"Learned Intercept: {model_lib.intercept_[0]}")

# Save the library model
joblib.dump(model_lib, os.path.join(MODELS_DIR, "logistic_regression_lib.joblib"))


Library Logistic Regression Training Accuracy: 70.86%
Learned Weights: [2.06157301 1.98318531]
Learned Intercept: -14.57125536898414


['..\\models\\logistic_regression_lib.joblib']

---
## 4. Algorithm Implementation 2: From Scratch (Pure NumPy)
To fulfill the core academic requirement of the SOP, we implement Logistic Regression from scratch using Gradient Descent. We avoid `sklearn` entirely for this step.


In [5]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def _sigmoid(self, z):
        # Clip to prevent overflow
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.epochs):
            # Forward pass
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)
            
            # Loss computation (Binary Cross Entropy)
            epsilon = 1e-9
            loss = -np.mean(y * np.log(y_predicted + epsilon) + (1 - y) * np.log(1 - y_predicted + epsilon))
            self.loss_history.append(loss)
            
            # Gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)
        
    def predict(self, X, threshold=0.5):
        y_predicted = self.predict_proba(X)
        return [1 if i > threshold else 0 for i in y_predicted]

print("Scratch Logistic Regression Class defined.")


Scratch Logistic Regression Class defined.


In [6]:
# Scale features for the scratch implementation (Gradient descent is sensitive to scaling)
X_train_scaled = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)

# Initialize and train scratch model
model_scratch = LogisticRegressionScratch(learning_rate=0.5, epochs=100)
print("Training scratch model...")
model_scratch.fit(X_train_scaled, y_train)

# Predict
y_pred_scratch = model_scratch.predict(X_train_scaled)
acc_scratch = accuracy_score(y_train, y_pred_scratch)

print(f"Scratch Logistic Regression Training Accuracy: {acc_scratch * 100:.2f}%")
print(f"Learned Weights: {model_scratch.weights}")
print(f"Learned Intercept: {model_scratch.bias}")


Training scratch model...
Scratch Logistic Regression Training Accuracy: 70.86%
Learned Weights: [0.89866753 0.75505151]
Learned Intercept: 0.20227546080839356


In [7]:
# Plot Loss History to prove Gradient Descent works
plt.figure(figsize=(8, 4))
plt.plot(model_scratch.loss_history, color='red', label="BCE Loss")
plt.title("Scratch Logistic Regression Convergence")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "03_scratch_loss_convergence.png")
plt.savefig(fig_path)
plt.close()
display(Image(filename=fig_path))



<Figure size 800x400 with 0 Axes>

---
## ✅ Week 3 Completion Checklist

- [x] Defined classification target (`is_liked` based on rating threshold).
- [x] Engineered continuous features suitable for regression models.
- [x] Trained model using `scikit-learn` (`LogisticRegression`).
- [x] Created `LogisticRegressionScratch` entirely from pure NumPy maths.
- [x] Plotted scratch gradient descent loss convergence graph.
